# HW9

This notebook builds weekly football event recommendations for June 2023. The main signal is the list of teams each user follows. If a user does not follow any team that plays in a given week, the notebook falls back to similar users from the same MCC group.

## Setup

The notebook uses the ClickHouse HTTP interface, so no extra database driver is needed. 
Set `CH_HOST`, `CH_PORT`, `CH_USER`, and optionally `CH_SCHEME` as environment variables before running it. 
The password is not stored in the notebook; if it is missing, the notebook will ask for it at runtime.

In [ ]:
from __future__ import annotations

import base64
import getpass
import io
import json
import os
import time
from collections import Counter, defaultdict
from dataclasses import dataclass
from urllib import request

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 140)
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
plt.style.use('seaborn-v0_8-whitegrid')

MCC_VALUES = [216, 218, 219, 220, 221, 222, 226, 232, 262, 276, 284, 293, 294, 297]
TOP_K = 20

CH_SCHEME = os.getenv('CH_SCHEME', 'https')
CH_HOST = os.getenv('CH_HOST', '')
CH_PORT = os.getenv('CH_PORT', '')
CH_USER = os.getenv('CH_USER', '')
CH_PASSWORD = os.getenv('CH_PASSWORD', '') or getpass.getpass('ClickHouse password: ')

missing = [name for name, value in {
    'CH_HOST': CH_HOST,
    'CH_PORT': CH_PORT,
    'CH_USER': CH_USER,
    'CH_PASSWORD': CH_PASSWORD,
}.items() if not value]
if missing:
    raise ValueError(f'Missing required configuration: {", ".join(missing)}')


## Helper functions

In [ ]:
def run_query(query: str) -> pd.DataFrame:
    url = f'{CH_SCHEME}://{CH_HOST}:{CH_PORT}/'
    req = request.Request(url, data=query.encode('utf-8'), method='POST')
    token = base64.b64encode(f'{CH_USER}:{CH_PASSWORD}'.encode('utf-8')).decode('ascii')
    req.add_header('Authorization', f'Basic {token}')
    with request.urlopen(req, timeout=120) as response:
        payload = response.read().decode('utf-8')
    return pd.read_csv(io.StringIO(payload), sep='\t')

def run_json_rows(query: str) -> list[dict]:
    url = f'{CH_SCHEME}://{CH_HOST}:{CH_PORT}/'
    req = request.Request(url, data=query.encode('utf-8'), method='POST')
    token = base64.b64encode(f'{CH_USER}:{CH_PASSWORD}'.encode('utf-8')).decode('ascii')
    req.add_header('Authorization', f'Basic {token}')
    with request.urlopen(req, timeout=120) as response:
        lines = response.read().decode('utf-8').strip().splitlines()
    return [json.loads(line) for line in lines if line]


## Queries

The brief mentions `bq.mobileuser`, but the available table in ClickHouse is `bq.mobile_user`, so I use that table below.

In [ ]:
TEAM_QUERY = '''
WITH team_rows AS (
    SELECT arrayJoin([hometeam_id, awayteam_id]) AS team_id
    FROM sports.event
    WHERE sport_id = 1
      AND startdate >= toDateTime('2023-01-01 00:00:00')
      AND startdate < toDateTime('2023-07-01 00:00:00')
)
SELECT DISTINCT team_id
FROM team_rows
ORDER BY team_id
FORMAT TabSeparatedWithNames
'''

USER_QUERY = f'''
SELECT
    user_account_id,
    teams,
    mcc,
    created_at,
    updated_at
FROM bq.mobile_user
WHERE user_account_id IS NOT NULL
  AND length(teams) > 0
  AND created_at <= toDateTime('2024-09-30 23:59:59')
  AND updated_at <= toDateTime('2024-09-30 23:59:59')
  AND mcc IN ({','.join(str(v) for v in MCC_VALUES)})
ORDER BY mcc, user_account_id, updated_at
FORMAT JSONEachRow
'''

EVENT_QUERY = '''
SELECT
    toDate(toMonday(startdate)) AS week_start,
    id AS event_id,
    startdate,
    hometeam_id,
    awayteam_id
FROM sports.event
WHERE sport_id = 1
  AND startdate >= toDateTime('2023-06-01 00:00:00')
  AND startdate < toDateTime('2023-07-01 00:00:00')
ORDER BY week_start, startdate, event_id
FORMAT TabSeparatedWithNames
'''


## Task 1. Teams

In [ ]:
teams_df = run_query(TEAM_QUERY)
teams_df.head()


In [ ]:
print('Relevant football teams:', len(teams_df))
teams_df.sample(10, random_state=42).sort_values('team_id')


## Task 1. Users

The raw table contains multiple snapshots for some `user_account_id` values, so I keep the latest row per account using `updated_at`. After that, I retain only followed teams that appear in the relevant football team set.

In [ ]:
raw_users = pd.DataFrame(run_json_rows(USER_QUERY))
raw_users['updated_at'] = pd.to_datetime(raw_users['updated_at'])
raw_users['created_at'] = pd.to_datetime(raw_users['created_at'])
print('Raw user rows:', len(raw_users))
print('Unique user_account_id before dedup:', raw_users['user_account_id'].nunique())
raw_users.head()


In [ ]:
relevant_teams = set(teams_df['team_id'])

users_df = (
    raw_users.sort_values(['user_account_id', 'updated_at'])
    .drop_duplicates(subset='user_account_id', keep='last')
    .sort_values(['mcc', 'user_account_id'])
    .reset_index(drop=True)
)
users_df['filtered_teams'] = users_df['teams'].apply(
    lambda team_list: sorted(set(team for team in team_list if team in relevant_teams))
)
users_df = users_df[users_df['filtered_teams'].map(bool)].copy()
users_df['team_set'] = users_df['filtered_teams'].apply(set)
users_df['num_followed_teams'] = users_df['filtered_teams'].str.len()

print('Users after dedup:', len(users_df))
print('Users after filtering to relevant teams:', len(users_df))
users_df[['user_account_id', 'filtered_teams', 'mcc', 'num_followed_teams']].head()


In [ ]:
users_by_mcc = users_df.groupby('mcc').agg(
    users=('user_account_id', 'count'),
    avg_followed_teams=('num_followed_teams', 'mean'),
    median_followed_teams=('num_followed_teams', 'median'),
).reset_index()
users_by_mcc


The user base is not evenly distributed across MCC groups, so keeping the recommendation step inside each MCC helps keep the neighbor search local and avoids mixing very different segments.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(users_by_mcc['mcc'].astype(str), users_by_mcc['users'], color='#4C72B0')
ax.set_title('Users by MCC')
ax.set_xlabel('MCC')
ax.set_ylabel('Users')
plt.tight_layout()
plt.show()


## Task 1. Events

Recommendations are created on a weekly basis for June 2023. I keep the event rows in a flat table because that is more useful for recommendation logic, then I also build a grouped weekly summary.

In [ ]:
events_df = run_query(EVENT_QUERY)
events_df['week_start'] = pd.to_datetime(events_df['week_start'])
events_df['startdate'] = pd.to_datetime(events_df['startdate'])
events_df.head()


In [ ]:
weekly_events_summary = (
    events_df.assign(team_pair=list(zip(events_df['hometeam_id'], events_df['awayteam_id'])))
    .groupby('week_start')
    .agg(
        event_count=('event_id', 'count'),
        event_ids=('event_id', list),
        team_ids=('team_pair', list),
    )
    .reset_index()
)
weekly_events_summary


## Task 2. Recommendation method

The recommendation flow is simple:

1. For each week, recommend every event where the user follows the home or away team.
2. If the user has no such events that week, look for similar users inside the same MCC group.
3. Similarity is Jaccard similarity over the filtered follow-team sets.
4. Take the top 10 neighbors with direct weekly recommendations and score events by the sum of neighbor similarities.
5. If there is no team overlap with any candidate in the MCC group, fall back to the most common weekly events inside that MCC group. If even that pool is empty, fall back to global weekly popularity.

For speed, the notebook keeps users grouped by `mcc`, stores followed teams as sets, and builds dictionary lookups from team to users and from team to weekly events. That keeps the similarity step local to the relevant MCC group and avoids scanning the full event list for every fallback user.

In [ ]:
@dataclass
class WeeklyEvent:
    event_id: int
    startdate: pd.Timestamp
    home_team: int
    away_team: int

weekly_events = defaultdict(list)
for row in events_df.itertuples(index=False):
    weekly_events[row.week_start].append(
        WeeklyEvent(
            event_id=int(row.event_id),
            startdate=row.startdate,
            home_team=int(row.hometeam_id),
            away_team=int(row.awayteam_id),
        )
    )

user_groups = {}
for mcc, group in users_df.groupby('mcc', sort=True):
    team_to_users = defaultdict(set)
    user_team_sets = {}
    for row in group.itertuples(index=False):
        user_team_sets[row.user_account_id] = row.team_set
        for team in row.team_set:
            team_to_users[team].add(row.user_account_id)
    user_groups[int(mcc)] = {
        'user_ids': list(group['user_account_id']),
        'team_to_users': team_to_users,
        'user_team_sets': user_team_sets,
    }


In [ ]:
def build_recommendations_for_week(week_start, week_items, user_groups, top_k=TOP_K):
    week_team_to_events = defaultdict(list)
    event_start_lookup = {item.event_id: item.startdate for item in week_items}
    for item in week_items:
        week_team_to_events[item.home_team].append(item.event_id)
        week_team_to_events[item.away_team].append(item.event_id)

    recommendation_rows = []
    global_direct_recs = {}
    global_popularity = Counter()

    for mcc, group_info in user_groups.items():
        user_ids = group_info['user_ids']
        team_to_users = group_info['team_to_users']
        user_team_sets = group_info['user_team_sets']

        direct_users = set()
        direct_recs = {}
        mcc_popularity = Counter()

        for user_id in user_ids:
            team_set = user_team_sets[user_id]
            recs = []
            for team in team_set:
                recs.extend(week_team_to_events.get(team, []))
            recs = sorted(set(recs), key=lambda event_id: (event_start_lookup[event_id], event_id))
            if recs:
                direct_recs[user_id] = recs
                direct_users.add(user_id)
                mcc_popularity.update(recs)
                global_direct_recs[user_id] = recs
                global_popularity.update(recs)

        for user_id in user_ids:
            team_set = user_team_sets[user_id]
            if user_id in direct_recs:
                recommendation_rows.append({
                    'week_start': week_start,
                    'mcc': mcc,
                    'user_account_id': user_id,
                    'source': 'followed_teams',
                    'event_ids': direct_recs[user_id],
                    'event_count': len(direct_recs[user_id]),
                })
                continue

            overlap_candidates = set()
            for team in team_set:
                overlap_candidates.update(team_to_users.get(team, set()))
            overlap_candidates &= direct_users

            event_scores = Counter()
            source = 'similar_users_same_mcc'

            if overlap_candidates:
                ranked = []
                for candidate_id in overlap_candidates:
                    candidate_teams = user_team_sets[candidate_id]
                    intersection = len(team_set & candidate_teams)
                    union = len(team_set | candidate_teams)
                    score = intersection / union if union else 0.0
                    ranked.append((score, -abs(len(team_set) - len(candidate_teams)), candidate_id))
                ranked.sort(reverse=True)
                top_neighbors = ranked[:10]
                for score, _, candidate_id in top_neighbors:
                    for event_id in direct_recs[candidate_id]:
                        event_scores[event_id] += score
            elif direct_users:
                source = 'weekly_popular_events_same_mcc'
                for event_id, count in mcc_popularity.most_common(top_k):
                    event_scores[event_id] = count
            else:
                source = 'weekly_popular_events_global'
                for event_id, count in global_popularity.most_common(top_k):
                    event_scores[event_id] = count

            ordered_events = [
                event_id
                for event_id, _ in sorted(
                    event_scores.items(),
                    key=lambda item: (-item[1], event_start_lookup[item[0]], item[0]),
                )[:top_k]
            ]
            recommendation_rows.append({
                'week_start': week_start,
                'mcc': mcc,
                'user_account_id': user_id,
                'source': source,
                'event_ids': ordered_events,
                'event_count': len(ordered_events),
            })

    return pd.DataFrame(recommendation_rows)


## Build weekly recommendations

In [ ]:
start = time.perf_counter()
weekly_recommendation_frames = []
coverage_rows = []

for week_start, week_items in weekly_events.items():
    week_df = build_recommendations_for_week(week_start, week_items, user_groups)
    weekly_recommendation_frames.append(week_df)
    coverage_rows.append({
        'week_start': week_start,
        'users': len(week_df),
        'users_with_events': int((week_df['event_count'] > 0).sum()),
        'follow_based': int((week_df['source'] == 'followed_teams').sum()),
        'fallback': int((week_df['source'] != 'followed_teams').sum()),
    })

recommendations_df = pd.concat(weekly_recommendation_frames, ignore_index=True)
coverage_df = pd.DataFrame(coverage_rows).sort_values('week_start').reset_index(drop=True)
elapsed_seconds = time.perf_counter() - start
elapsed_seconds


## Coverage check

All users should receive recommendations. The table below checks that every user-week has at least one event.

In [ ]:
coverage_df


In [ ]:
print('All user-weeks covered:', bool((coverage_df['users'] == coverage_df['users_with_events']).all()))
print('Total recommendation rows:', len(recommendations_df))
print('Unique users covered:', recommendations_df['user_account_id'].nunique())
print('Average number of recommended events:', recommendations_df['event_count'].mean())


The coverage check is the main sanity test here. All user-weeks should be covered, and the split between direct and fallback recommendations shows how often the similarity step is actually needed.

In [ ]:
coverage_plot = coverage_df[['week_start', 'follow_based', 'fallback']].copy()
coverage_plot['week_label'] = coverage_plot['week_start'].dt.strftime('%Y-%m-%d')

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(coverage_plot['week_label'], coverage_plot['follow_based'], label='follow_based', color='#4C72B0')
ax.bar(coverage_plot['week_label'], coverage_plot['fallback'], bottom=coverage_plot['follow_based'], label='fallback', color='#DD8452')
ax.set_title('Weekly recommendation sources')
ax.set_xlabel('Week start')
ax.set_ylabel('Users')
ax.legend()
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## Recommendation examples

In [ ]:
recommendations_df.head(10)


In [ ]:
recommendations_df['source'].value_counts().to_frame('user_weeks')


In [ ]:
sample_fallbacks = recommendations_df[recommendations_df['source'] != 'followed_teams'].head(10)
sample_fallbacks


## Final notes

This solution uses a direct follow-based rule first because that is the strongest available signal in the task. For users without direct weekly matches, it switches to a lightweight user-based collaborative step inside the same MCC group. That keeps the method simple, fast, and aligned with the brief.